# Harvest ECOSOC summary-record PDFs

Downloads `E/{year}/SR.{n}` PDFs from `documents.un.org`, extracts text with `pypdf` (fallback `pdfplumber`), and writes `data/interim/transcripts.parquet`.

## 0. Setup

In [3]:
!pip install requests pypdf pdfplumber pandas pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 12.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 10.1 MB/s eta 0:00:00a 0:00:01


In [4]:
import csv
import hashlib
import logging
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
import requests
from pypdf import PdfReader

URL_TEMPLATE = "https://documents.un.org/api/symbol/access?s=E/{year}/SR.{n}&l=en&t=pdf"
USER_AGENT = "ecosoc-research-harvester/0.1 (lwilliams127@uchicago.edu)"
MIN_PDF_BYTES = 5_000
SLEEP_SECONDS = 1.0
MAX_CONSECUTIVE_MISSES = 3
REQUEST_TIMEOUT = 30

ROOT = Path.cwd().parent  # notebooks/ -> ecosoc/
PDF_DIR = ROOT / "data" / "raw" / "ecosoc_pdfs"
OUT_PARQUET = ROOT / "data" / "interim" / "transcripts.parquet"
LOG_PATH = ROOT / "data" / "raw" / "_harvest_log.csv"

PDF_DIR.mkdir(parents=True, exist_ok=True)
OUT_PARQUET.parent.mkdir(parents=True, exist_ok=True)

logging.basicConfig(format="%(asctime)s %(levelname)s %(message)s",
                    level=logging.INFO, datefmt="%H:%M:%S")
log = logging.getLogger("harvest")
print("ROOT:", ROOT)

ROOT: /Users/latahviawilliams/Downloads/Big_Data_export/final_project/ecosoc


## 1. Helpers

In [5]:
def _sha256(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()

def _pdf_path(year: int, n: int) -> Path:
    return PDF_DIR / f"E_{year}_SR_{n:03d}.pdf"

def fetch_one(session: requests.Session, year: int, n: int) -> str:
    """Returns 'saved', 'skip', 'miss', or 'error'."""
    out = _pdf_path(year, n)
    if out.exists() and out.stat().st_size >= MIN_PDF_BYTES:
        return "skip"

    url = URL_TEMPLATE.format(year=year, n=n)
    for attempt in range(3):
        try:
            r = session.get(url, timeout=REQUEST_TIMEOUT, allow_redirects=True)
        except requests.RequestException as e:
            log.warning("net error %s y=%s n=%s attempt=%s", e, year, n, attempt)
            time.sleep(2 ** attempt)
            continue

        if r.status_code == 404:
            return "miss"
        if r.status_code != 200:
            log.warning("http %s y=%s n=%s", r.status_code, year, n)
            time.sleep(2 ** attempt)
            continue

        ctype = r.headers.get("Content-Type", "").lower()
        if "pdf" not in ctype or len(r.content) < MIN_PDF_BYTES:
            return "miss"

        out.write_bytes(r.content)
        return "saved"
    return "error"

In [6]:
def extract_text(pdf_path: Path) -> tuple[str, int]:
    """pypdf first; fall back to pdfplumber if extracted text < 200 chars."""
    text, n_pages = "", 0
    try:
        reader = PdfReader(str(pdf_path))
        n_pages = len(reader.pages)
        text = "\n".join((p.extract_text() or "") for p in reader.pages)
    except Exception as e:
        log.warning("pypdf failed %s: %s", pdf_path.name, e)

    if len(text) < 200:
        try:
            import pdfplumber
            with pdfplumber.open(str(pdf_path)) as pdf:
                n_pages = len(pdf.pages)
                text = "\n".join((p.extract_text() or "") for p in pdf.pages)
        except ImportError:
            log.info("pdfplumber not installed; skipping fallback for %s", pdf_path.name)
        except Exception as e:
            log.warning("pdfplumber failed %s: %s", pdf_path.name, e)
    return text, n_pages

## 2. Configure the run

For a quick smoke test, run a single year first (e.g. 2023). For the full corpus, set `START_YEAR=2000`, `END_YEAR=datetime.utcnow().year`.

In [7]:
START_YEAR = 2000
END_YEAR = datetime.utcnow().year  # 2026
MAX_MEETINGS = 75
SKIP_DOWNLOAD = False  # set True to re-extract text from existing PDFs only

/var/folders/bp/n09_8ybn2t3bm81y47gj3vg00000gn/T/ipykernel_58413/1604465220.py:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  END_YEAR = datetime.utcnow().year  # 2026


## 3. Download loop

In [8]:
if not SKIP_DOWNLOAD:
    session = requests.Session()
    session.headers.update({"User-Agent": USER_AGENT, "Accept": "application/pdf"})
    new_log_rows = []
    for year in range(START_YEAR, END_YEAR + 1):
        misses = 0
        for n in range(1, MAX_MEETINGS + 1):
            status = fetch_one(session, year, n)
            ts = datetime.utcnow().isoformat(timespec="seconds")
            new_log_rows.append({"ts": ts, "year": year, "n": n, "status": status})
            log.info("y=%s n=%s -> %s", year, n, status)
            if status == "miss":
                misses += 1
                if misses >= MAX_CONSECUTIVE_MISSES:
                    log.info("y=%s: %s consecutive misses, stopping year", year, misses)
                    break
            else:
                misses = 0
            if status == "saved":
                time.sleep(SLEEP_SECONDS)

    write_header = not LOG_PATH.exists()
    with LOG_PATH.open("a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["ts", "year", "n", "status"])
        if write_header:
            w.writeheader()
        w.writerows(new_log_rows)
    print(f"download done; {sum(1 for r in new_log_rows if r['status']=='saved')} saved, "
          f"{sum(1 for r in new_log_rows if r['status']=='skip')} skipped")

/var/folders/bp/n09_8ybn2t3bm81y47gj3vg00000gn/T/ipykernel_58413/3910110317.py:9: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow().isoformat(timespec="seconds")
17:05:39 INFO y=2000 n=1 -> miss
17:05:40 INFO y=2000 n=2 -> miss
17:05:42 INFO y=2000 n=3 -> saved
17:05:44 INFO y=2000 n=4 -> saved
17:05:45 INFO y=2000 n=5 -> miss
17:05:45 INFO y=2000 n=6 -> miss
17:05:46 INFO y=2000 n=7 -> miss
17:05:46 INFO y=2000: 3 consecutive misses, stopping year
17:05:47 INFO y=2001 n=1 -> saved
17:05:49 INFO y=2001 n=2 -> saved
17:05:52 INFO y=2001 n=3 -> saved
17:05:54 INFO y=2001 n=4 -> saved
17:05:57 INFO y=2001 n=5 -> saved
17:05:59 INFO y=2001 n=6 -> saved
17:06:02 INFO y=2001 n=7 -> saved
17:06:05 INFO y=2001 n=8 -> saved
17:06:08 INFO y=2001 n=9 -> saved
17:06:11 INFO y=2001 n=10 -> saved
17:06:12 INFO y=2001 n=11 ->

download done; 910 saved, 0 skipped


## 4. Extract text → parquet

In [9]:
rows = []
pdfs = sorted(PDF_DIR.glob("E_*_SR_*.pdf"))
print(f"extracting from {len(pdfs)} PDFs")
for p in pdfs:
    try:
        _, year_s, _, n_s = p.stem.split("_")
        year, n = int(year_s), int(n_s)
    except ValueError:
        log.warning("unparseable filename %s", p.name)
        continue
    text, n_pages = extract_text(p)
    rows.append({
        "year": year,
        "meeting_n": n,
        "meeting_id": f"E/{year}/SR.{n}",
        "segment": "SR",
        "n_pages": n_pages,
        "text": text,
        "n_chars": len(text),
        "sha256": _sha256(p.read_bytes()),
        "source_pdf": p.name,
    })

df = pd.DataFrame(rows)
df.to_parquet(OUT_PARQUET, index=False)
print(f"wrote {OUT_PARQUET} ({len(df)} rows)")
df.head()

extracting from 910 PDFs


17:44:43 WARNING invalid pdf header: b'\x00\x00\x00\x00\x00'
17:44:43 WARNING EOF marker not found
17:44:43 WARNING pypdf failed E_2001_SR_010.pdf: Stream has ended unexpectedly
17:44:44 WARNING pdfplumber failed E_2001_SR_010.pdf: No /Root object! - Is this really a PDF?
17:44:44 WARNING invalid pdf header: b'\x00\x00\x00\x00\x00'
17:44:44 WARNING EOF marker not found
17:44:44 WARNING pypdf failed E_2001_SR_015.pdf: Stream has ended unexpectedly
17:44:44 WARNING pdfplumber failed E_2001_SR_015.pdf: No /Root object! - Is this really a PDF?
17:44:45 WARNING invalid pdf header: b'\x00\x00\x00\x00\x00'
17:44:45 WARNING EOF marker not found
17:44:45 WARNING pypdf failed E_2001_SR_036.pdf: Stream has ended unexpectedly
17:44:45 WARNING pdfplumber failed E_2001_SR_036.pdf: No /Root object! - Is this really a PDF?
17:44:46 WARNING invalid pdf header: b'\x00\x00\x00\x00\x00'
17:44:46 WARNING EOF marker not found
17:44:46 WARNING pypdf failed E_2001_SR_043.pdf: Stream has ended unexpectedly
17:

wrote /Users/latahviawilliams/Downloads/Big_Data_export/final_project/ecosoc/data/interim/transcripts.parquet (910 rows)


,year,meeting_n,meeting_id,segment,n_pages,text,n_chars,sha256,source_pdf
0,2000,3,E/2000/SR.3,SR,5,United Nations E/2000/SR.3\n \nEconomic and So...,15766,ca2e96cf5594b54ec97f86acfe714aaaffd8a01d73d665...,E_2000_SR_003.pdf
1,2000,4,E/2000/SR.4,SR,8,United Nations E/2000/SR.4\n \nEconomic and So...,32904,893f98840d4f8cba760f618c98672470206f954c79ea63...,E_2000_SR_004.pdf
2,2001,1,E/2001/SR.1,SR,6,United Nations E/2001/SR.1\n \nEconomic and So...,21294,f553086805c3b54f7ec3a2e964ffc9f968db6a17111489...,E_2001_SR_001.pdf
3,2001,2,E/2001/SR.2,SR,5,United Nations E/2001/SR.2\n \nEconomic and So...,16143,edc68809a772cb221108298349e5fabdefaef8c5117f96...,E_2001_SR_002.pdf
4,2001,3,E/2001/SR.3,SR,6,United Nations E/2001/SR.3\n \nEconomic and So...,20828,9e8719f748063fedb6baf6e5e52c69e90e6d0b711cefe2...,E_2001_SR_003.pdf


## 5. Quick QA

In [10]:
if len(df):
    print("rows:", len(df))
    print("years:", df.year.min(), "-", df.year.max())
    print("per-year counts:")
    print(df.groupby('year').size())
    print("\nempty-text rows:", int((df.n_chars < 200).sum()))
    print("\nsample text head (first non-empty):")
    sample = df[df.n_chars >= 500].head(1)
    if len(sample):
        print(sample.iloc[0].text[:800])

rows: 910
years: 2000 - 2025
per-year counts:
year
2000     2
2001    36
2002    44
2003    45
2004    54
2005    41
2006    47
2007    43
2008    48
2009    47
2010    15
2011    47
2012    16
2013    52
2014     9
2015    54
2016    49
2017    49
2018    49
2019    22
2020     6
2021    13
2022    34
2023    44
2024    39
2025     5
dtype: int64

empty-text rows: 4

sample text head (first non-empty):
United Nations E/2000/SR.3
 
Economic and Social Council Provisional
17 August 2000
English
Original: French
Corrections to this record should be submitted in one of the working languages. They should be
set forth in a memorandum and also incorporated in a copy of the record. They should be sent
within one week of the date of this document  to the Chief, Official Records Editing Section,
room DC2-750, 2 United Nations Plaza.
00-28009 (E)
`````````
Organizational session for 2000
Provisional summary record of the 3rd meeting
Held at Headquarters, New York, on Friday, 4 February 2000, at 